# Reproducing Tingley et al. (2021) from dandiset 000233

**Paper.** Tingley D, McClain K, Kaya E, Carpenter J, Buzsáki G (2021). A metabolic function of
the hippocampal sharp wave-ripple. *Nature* 597, 82-86.
[doi:10.1038/s41586-021-03811-w](https://doi.org/10.1038/s41586-021-03811-w). The claims file was
written from the accepted author manuscript in PubMed Central (PMC9214835) with its Methods,
Extended Data legends and Supplementary Information.

**Dandiset.** [DANDI:000233](https://dandiarchive.org/dandiset/000233/0.250624.0414), version
0.250624.0414. The paper cites no DANDI version (its data statement points to the lab's server).
This is the most recent published version; the earlier one, 0.230223.0815, lists the same 345
files with the same sizes, so the choice does not change the data.

**What the dandiset holds.** 345 NWB files from 25 rats: CA1 LFP at 1,250 Hz, tables of detected
SPW-Rs, the iPro2 glucose sensor current, a head accelerometer, and sleep states for six rats. It
holds no spike units, no optogenetic stimulation times, no CNO or vehicle labels and injection
times, no meal times, and no cohort labels, so the optogenetic, DREADD, fasting and unit claims
cannot be tested.

Random numbers are drawn only for circular shuffles, with a fixed seed in each claim's section.

| Claim | Status |
|---|---|
| fig2-ripple-rate-predicts-glucose | tested |
| fig2ef-correlation-beyond-shuffle | tested |
| fig2g-clustered-ripples | tested |
| edfig4-ripple-features | tested |
| fig2h-phase-reset | tested |
| edfig5-cross-frequency-coupling | tested |
| edfig1-glucose-rhythm | tested |
| edfig1-glucose-circadian-meals | blocked: meal times not in the dandiset (circadian part run) |
| edfig1-sensor-decay | tested |
| supp-ripple-rate | tested |
| supp-ripples-stable-across-days | tested |
| edfig3-ripple-rate-rhythms | tested |
| edfig2-refractory-period | tested |
| edfig4-nrem-wake | tested |
| edfig4-no-hour-differs | tested |
| edfig9-dorsal-ventral | blocked: one rat with dorsal and ventral SPW-Rs and glucose together (run) |
| edfig3e-ca1-firing-decrease, edfig8-lateral-septum-rectified | blocked: no spike units |
| edfig4-fasting-fed | blocked: no meal times |
| edfig6-ripple-rate-best-predictor, edfig6d-pairwise-predictors, edfig6ef-latent-variables | blocked: too large to compute the predictors (timed below) |
| fig3d, fig3-induced-effect-size, fig3ef, fig3f, edfig7ab, edfig7jm, edfig7-ppc | blocked: no stimulation times |
| fig4-cno-controls-no-change, fig4-ls-dreadd-decouples, fig4-ls-dreadd-chance-level, fig4e-dreadd-time-course, fig4c-medial-septum-no-effect, edfig10-expression-correlates-effect | blocked: no CNO or vehicle labels and no cohort labels |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats

import sprglu as S

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()

In [ ]:
assets = S.list_assets()
meta = S.version_metadata()
files = S.read_all(assets)
# the glucose series is stored in every file; all are read to check that they agree
glu_all = S.read_all_glucose(assets)
print(f"dandiset {S.DANDISET} version {S.VERSION}: {len(assets)} assets, "
      f"{assets['size'].sum() / 1e12:.2f} TB; read in {time.time() - started:.0f} s")

## Data check

The archive's metadata and the files are checked against each other and against the paper's
description. Each check is stated under its output.

In [ ]:
print("Archive summary:", meta["assetsSummary"]["numberOfSubjects"], "subjects,",
      meta["assetsSummary"]["numberOfFiles"], "files;", meta["assetsSummary"]["variableMeasured"])
fi = pd.DataFrame([{
    "subject": f["subject"], "path": f["path"].split("/")[1], "experiment": f["experiment"][:40],
    "lfp_t0": f["lfp_t0"], "hours": f["lfp_n"] / f["lfp_rate"] / 3600, "lfp_rate": f["lfp_rate"],
    "channels": f["lfp_channels"], "acquisition": "+".join(f["acquisition"]),
    "tables": "+".join(sorted(f["tables"])), "units": f["units"],
    "sleep_states": f["sleep"] is not None and len(f["sleep"]) > 0} for f in files])
print("LFP rates:", sorted(fi.lfp_rate.unique()), " units tables in any file:", fi.units.any())
print("experiment_description values:", fi.experiment.value_counts().to_dict())
print("acquisition series:", fi.acquisition.value_counts().to_dict())
print("SPW-R tables per file:", fi.tables.value_counts().to_dict())

No file holds spike units, optogenetic stimulation times, injection times or labels (CNO or
vehicle), meal times, or a cohort label: only the files of CGM1 name their experiment
("ripple/glucose recording", surgery rCA1), the rest point to Supplementary Table 1, whose rats
are numbered 1 to 45 while the dandiset's subjects are named CGM1 to CGM60, bruce, dt15, flex1,
ros and Vanessa, with no mapping between the two. SPW-R tables come under seven names; the
dorsal and ventral tables identify the dorsal and ventral rats.

In [ ]:
dups = S.find_duplicates(files)
drop = {i for g in dups for i in g[1:]}
print(f"{len(dups)} recordings are deposited twice (same subject, same LFP start and sample count):")
dd = pd.DataFrame([{"subject": files[g[0]]["subject"], "file_1": files[g[0]]["path"].split("/")[1],
                    "file_2": files[g[1]]["path"].split("/")[1],
                    "hours": files[g[0]]["lfp_n"] / 1250 / 3600,
                    "same_events": all(np.array_equal(files[g[0]]["tables"][k].peak.values,
                                                      files[g[1]]["tables"][k].peak.values)
                                       for k in files[g[0]]["tables"] if k in files[g[1]]["tables"])}
                   for g in dups])
print(dd.round(2).to_string())
kept = [f for i, f in enumerate(files) if i not in drop]
per_subject = fi.drop(index=list(drop)).groupby("subject").agg(
    files=("path", "size"), hours=("hours", "sum"), with_sleep_states=("sleep_states", "sum"))
print(per_subject.round(1).to_string())
print("subjects:", len(per_subject), " files counted once:", len(kept), " hours of LFP:", round(per_subject.hours.sum(), 1))

27 recordings are deposited twice, under a file name with dashes and one with underscores
(subjects CGM1, CGM3, CGM30, CGM31, CGM32); both copies have the same SPW-R events. Each is
counted once from here on.

In [ ]:
rows = []
for f in kept:
    named = S.file_start_from_name(f["path"])
    on_clock = S.reference_time(f["reference"]) + pd.Timedelta(seconds=f["lfp_t0"])
    row = {"subject": f["subject"], "name_minus_clock_s": (on_clock - named).total_seconds() if named else np.nan}
    for k, t in f["tables"].items():
        if len(t):
            row["start_minus_t0_plus_peak_s"] = float(np.median(t.start_time - (f["lfp_t0"] + t.peak)))
            row["peak_max_minus_duration_s"] = float(t.peak.max() - f["lfp_n"] / f["lfp_rate"])
    rows.append(row)
ck = pd.DataFrame(rows)
print("files whose name has no date:", ck.name_minus_clock_s.isna().sum(),
      "; |file-name time - session clock| max (s):", ck.name_minus_clock_s.abs().max())
print("SPW-R start_time - (LFP start + peak), s: ", ck.start_minus_t0_plus_peak_s.describe()[["min", "50%", "max"]].round(4).to_dict())
print("last SPW-R peak - file duration, s (<= 0 means peaks are within the file):",
      round(ck.peak_max_minus_duration_s.max(), 3))
ev_all = pd.concat([t.assign(subject=f["subject"], table=k) for f in kept for k, t in f["tables"].items()])
print("share of stored CA1 events below 5 s.d. per subject:",
      {s_: round(float((g.peak_normed_power < 5).mean()), 3) for s_, g in ev_all[ev_all.table.isin(S.CA1_TABLES)].groupby("subject")})
print(ev_all.groupby("table").agg(events=("peak", "size"), min_normed_power=("peak_normed_power", "min"),
      min_duration_ms=("duration", lambda x: 1000 * x.min()), max_duration_ms=("duration", lambda x: 1000 * x.max()),
      median_frequency=("peak_frequency", "median"), min_frequency=("peak_frequency", "min"),
      max_frequency=("peak_frequency", "max")).round(2).to_string())

The date and time in each file name equals the session reference time plus the LFP starting time,
so the session clock is the lab's wall clock (the reference is labelled UTC but holds local time).
In the SPW-R tables, `start_time` is on the session clock and `peak` is relative to the start of the
file (start_time is LFP start + peak minus 10 to 45 ms, the time from start to peak); the column
descriptions do not say this. SPW-R times used here are LFP start + peak. Durations lie in 20 to 200
ms, inside the Methods' 15 to 250 ms. The Methods set the peak threshold at 5 s.d.; the stored
peak normed power is at least 5 in the tables of 12 subjects but goes down to 4 in nine others
(7 to 29% of their events) and below 1 in CGM30, so those tables were made with a lower threshold.
The analysis keeps events of at least 5 s.d. (Methods) and shows the main results with all stored
events as a variant.

In [ ]:
ss = pd.concat([f["sleep"].assign(subject=f["subject"]) for f in kept if f["sleep"] is not None])
print("sleep-state labels:", ss.label.value_counts().to_dict(), " subjects:", sorted(ss.subject.unique()))
# clock check between SPW-Rs and sleep states: SPW-R rate per state, true and with states shifted by 30 min
rows = []
for f in kept:
    if f["sleep"] is None or not len(f["sleep"]) or S.ca1_table(f) is None:
        continue
    e = S.ripple_events(f).t.values
    for shift in (0, 1800):
        for lab, g in f["sleep"].groupby("label"):
            st, sp = g.start.values + shift, g.stop.values + shift
            n = sum(((e >= a) & (e < b)).sum() for a, b in zip(st, sp))
            rows.append({"subject": f["subject"], "shift_s": shift, "state": lab, "events": n, "s": (sp - st).sum()})
r = pd.DataFrame(rows).groupby(["subject", "shift_s", "state"]).sum()
r["per_min"] = 60 * r.events / r.s
print(r.per_min.unstack("state").round(2).to_string())

Sleep states are present for six subjects. In CGM1, CGM2, CGM3 and flex1, SPW-Rs occur at 18 to
21 per minute in NREM sleep against 3 to 12 when awake with the states as stored, and the contrast
shrinks when the states are shifted by 30 min, so the SPW-R tables and the sleep states share a
clock there. In dt15 the contrast is small (5.9 against 5.5 per minute) and the check does not
establish it; bruce has no SPW-R table.

In [ ]:
rows, GLU_FIRST = [], {}
for s, idx in assets.groupby("subject").groups.items():
    stored = {(glu_all[i]["t"].tobytes(), glu_all[i]["nA"].tobytes()) for i in idx}
    wall = {tuple(sorted(zip(S.clock(files[i]["reference"], glu_all[i]["t"]), glu_all[i]["nA"]))) for i in idx}
    refs = {files[i]["reference"] for i in idx}
    rows.append({"subject": s, "files": len(idx), "distinct_series_as_stored": len(stored),
                 "distinct_series_on_wall_clock": len(wall), "session_references": len(refs)})
print(pd.DataFrame(rows).to_string())
for f, g in zip(files, glu_all):
    GLU_FIRST.setdefault(f["subject"], g)
rows = []
for s in sorted(GLU_FIRST):
    g = GLU_FIRST[s]
    ref = next(f["reference"] for f in files if f["subject"] == s)
    t = np.sort(g["t"])
    runs, swapped, b_stored, b_swapped = S.repaired_glucose(ref, g)
    rows.append({"subject": s, "readings": len(g["t"]),
                 "unit": g["unit"], "nA_min": g["nA"].min(), "nA_median": np.median(g["nA"]), "nA_max": g["nA"].max(),
                 "share_steps_300s": np.mean(np.abs(np.diff(t) - 300) < 10), "order_breaks_as_stored": int((np.abs(np.diff(g["t"]) - 300) > 10).sum()),
                 "breaks_sorted_as_stored": b_stored, "breaks_day_month_swapped": b_swapped, "repaired": swapped,
                 "first_reading": str(S.clock(ref, [runs.t.min()])[0])[:16], "last_reading": str(S.clock(ref, [runs.t.max()])[0])[:16]})
gq = pd.DataFrame(rows)
print(gq.round(3).to_string())
print("glucose description:", GLU_FIRST["CGM1"]["description"])

Each subject's glucose series is stored whole in every one of its files. In CGM1, CGM2 and CGM3
one file stores it in a different order or relative to a different session reference time (CGM2
and CGM3 have two reference times), but on the wall clock every file holds the same readings. Times
are therefore put on one clock per subject, that of its first file (helper `clock_offset`); the
series of the first file is used. Readings are 5 min apart (300 s; a few at 301 and 302 s). The values are
called nA, and their range (about 0 to 120) is consistent with the sensor current. In twelve
subjects the timestamps are out of order and the series jumps by about a month at each midnight:
dates whose day is 12 or less were stored with day and month swapped (a reading of 4 March 2021 is
stored as 3 April). Swapping them back joins the pieces into runs of consecutive 5-min readings
(fewer breaks, table above) that overlap the electrophysiology. The repair is applied to a subject
only when it reduces the breaks; the effect of the repair on the SPW-R and glucose correlation is
shown below.

In [ ]:
SUBJ = {}
rows = []
for s in sorted(GLU_FIRST):
    fs = [f for f in kept if f["subject"] == s]
    ref = next(f["reference"] for f in files if f["subject"] == s)
    ev, iv = S.subject_events(fs, reference=ref)
    ev_all, _ = S.subject_events(fs, min_peak_sd=0, reference=ref)
    glu, swapped, _, _ = S.repaired_glucose(ref, GLU_FIRST[s])
    b = S.binned(glu, ev, iv) if ev is not None else None
    b_all = S.binned(glu, ev_all, iv) if ev is not None else None
    stored = S.binned(S.glucose_runs(GLU_FIRST[s]["t"], GLU_FIRST[s]["nA"]), ev, iv) if ev is not None else None
    if b is not None:
        b["clock"] = S.clock(ref, b.t)
    SUBJ[s] = {"files": fs, "ref": ref, "ev": ev, "ev_all": ev_all, "iv": iv, "glu": glu, "b": b, "b_all": b_all, "stored": stored, "swapped": swapped}
    ok = 0 if b is None else int((np.isfinite(b.rate) & np.isfinite(b.delta)).sum())
    rows.append({"subject": s, "ca1_files": len(iv), "ca1_hours": sum(y - x for x, y in iv) / 3600,
                 "spw_r_stored": 0 if ev is None else len(ev_all), "spw_r_5sd": 0 if ev is None else len(ev), "glucose_hours": len(glu) * 5 / 60,
                 "hours_with_both": ok * 5 / 60,
                 "hours_with_both_as_stored": 0 if stored is None else int((np.isfinite(stored.rate) & np.isfinite(stored.delta)).sum()) * 5 / 60})
cov = pd.DataFrame(rows)
print(cov.round(1).to_string())
RATS = [s for s in cov[cov.hours_with_both > 0].subject]
print(len(RATS), "rats with CA1 SPW-Rs and glucose recorded together:", RATS,
      f"; {cov.hours_with_both.sum():.1f} h with both ({cov.hours_with_both_as_stored.sum():.1f} h with the timestamps as stored)")

Eighteen subjects have CA1 SPW-Rs and glucose recorded at the same time. In CGM36, bruce and ros
no file has a SPW-R table; in CGM37, CGM60, dt15 and flex1 the glucose readings and the
electrophysiology cover different days. Glucose and electrophysiology share no event that could
show their clocks agree independently of the claims; the repair of the dates, which brings the
two into overlap, is consistent with the file names but the alignment within a day is taken as
stored.

## fig2-ripple-rate-predicts-glucose

> SPW-R rate is negatively correlated with future changes in interstitial glucose, with a mean
> peak correlation of R = -0.21 (+- 0.08) at a lag of 10.7 (+- 4.5) min, over 30 rats.

Readings, as the claims file lists them: delta current from the raw readings (a) or after a
high-pass filter (b, a centred 24-h running mean subtracted within each run, a standard choice
for the cutoff the paper does not give); Pearson or Spearman correlation; one correlogram per rat
(a) or one per session averaged per rat (b; a session is a file, used when it holds at least 6 h
of bins, a standard choice); the lag searched over 0 to +100 min (a) or -100 to +100 min (b); all
rats with SPW-Rs and glucose together (a) or the ripple and glucose cohort (b; only CGM1 is
labelled as such in the files). The paper does not say how the SPW-R count of a 5-min bin is
aligned with the glucose change; the main analysis pairs the count and the change over the same
interval (an assumption), and the two alignments one reading earlier or later are shown as
variants.

In [ ]:
def highpass(b, hours=24):
    out = b.copy()
    w = int(hours * 12)
    out["nA"] = b.nA - b.groupby("run").nA.transform(lambda x: x.rolling(w, center=True, min_periods=1).mean())
    same = out.run.eq(out.run.shift(1))
    out["delta"] = np.where(same, out.nA - out.nA.shift(1), np.nan)
    return out

def per_session(b, iv, min_bins=72):
    """Rows of b inside each file, one table per file with at least min_bins bins of both."""
    out = []
    for x, y in iv:
        sel = b[(b.t - S.STEP >= x) & (b.t <= y)]
        if (np.isfinite(sel.rate) & np.isfinite(sel.delta)).sum() >= min_bins:
            out.append(sel)
    return out

LAG_MIN = S.LAGS * 5
CC = {}
for s in RATS:
    d = SUBJ[s]
    for dname, bb in (("raw", d["b"]), ("highpass", highpass(d["b"]))):
        for m in ("pearson", "spearman"):
            CC[(s, dname, m, "rat")] = S.xcorr(bb, method=m)
            sess = per_session(bb, d["iv"])
            CC[(s, dname, m, "session")] = (np.nanmean([S.xcorr(x, method=m) for x in sess], axis=0)
                                            if sess else np.full(len(S.LAGS), np.nan))

def peak(cc, lag_range):
    sel = LAG_MIN >= 0 if lag_range == "0..+100" else np.ones(len(LAG_MIN), bool)
    if np.all(np.isnan(cc[sel])):
        return np.nan, np.nan
    i = np.nanargmin(np.where(sel, cc, np.nan))
    return cc[i], LAG_MIN[i]

rows = []
for (s, dname, m, unit), cc in CC.items():
    for lr in ("0..+100", "-100..+100"):
        R, lag = peak(cc, lr)
        rows.append({"rat": s, "delta": dname, "corr": m, "unit": unit, "lags": lr, "R": R, "lag_min": lag,
                     "R_at_10": cc[LAG_MIN == 10][0]})
PEAKS = pd.DataFrame(rows)

In [ ]:
def summarise(df):
    return pd.Series({"rats": df.R.notna().sum(), "mean_R": df.R.mean(), "sd_R": df.R.std(),
                      "mean_lag": df.lag_min.mean(), "sd_lag": df.lag_min.std(),
                      "rats_R_negative": (df.R < 0).sum(), "R_ok": abs(df.R.mean() + 0.21) <= 0.08,
                      "lag_ok": abs(df.lag_min.mean() - 10.7) <= 4.5})
tab_a = PEAKS.groupby(["delta", "corr", "unit", "lags"]).apply(summarise)
tab_a.insert(0, "rats_set", "a: all")
tab_b = PEAKS[PEAKS.rat == "CGM1"].groupby(["delta", "corr", "unit", "lags"]).apply(summarise)
tab_b.insert(0, "rats_set", "b: CGM1")
T1 = pd.concat([tab_a, tab_b]).reset_index()
T1["both_hold_and_most_negative"] = T1.R_ok & T1.lag_ok & (T1.rats_R_negative > T1.rats / 2)
print(T1.round(3).to_string())

In [ ]:
main = PEAKS[(PEAKS["delta"] == "raw") & (PEAKS["corr"] == "spearman") & (PEAKS.unit == "rat") & (PEAKS.lags == "-100..+100")]
print("Per rat, raw delta, Spearman, one correlogram per rat, lags -100..+100 min:")
print(main.merge(cov[["subject", "hours_with_both"]], left_on="rat", right_on="subject")[["rat", "hours_with_both", "R", "lag_min", "R_at_10"]].round(3).to_string())
for minh in (6, 12, 24):
    keep = cov[cov.hours_with_both >= minh].subject
    x = PEAKS[PEAKS.rat.isin(keep)].groupby(["delta", "corr", "unit", "lags"]).apply(summarise)
    print(f"rats with at least {minh} h of both: combinations where both values hold: {int((x.R_ok & x.lag_ok).sum())} of {len(x)};"
          f" raw/spearman/rat/-100..+100: R {x.loc[('raw', 'spearman', 'rat', '-100..+100'), 'mean_R']:.3f},"
          f" lag {x.loc[('raw', 'spearman', 'rat', '-100..+100'), 'mean_lag']:.1f} min, rats {int(x.loc[('raw', 'spearman', 'rat', '-100..+100'), 'rats'])}")
# variant: other alignments of the SPW-R interval and the glucose change, one reading apart
for name, how in (("SPW-R rate over the 5 min after each reading", "rate"), ("delta to the next reading", "delta")):
    al = []
    for s in RATS:
        for dname in ("raw", "highpass"):
            bb = SUBJ[s]["b"] if dname == "raw" else highpass(SUBJ[s]["b"])
            bb = bb.copy()
            bb[how] = bb.groupby("run")[how].shift(-1)
            for m in ("pearson", "spearman"):
                cc = S.xcorr(bb, method=m)
                for lr in ("0..+100", "-100..+100"):
                    R, lag = peak(cc, lr)
                    al.append({"rat": s, "delta": dname, "corr": m, "lags": lr, "R": R, "lag_min": lag})
    t_al = pd.DataFrame(al).groupby(["delta", "corr", "lags"]).apply(summarise)
    print(f"Alignment variant, {name} (one correlogram per rat, all rats):")
    print(t_al.round(3).to_string())
# variant: all stored events, without the Methods' 5 s.d. threshold
va = []
for s in RATS:
    for m in ("pearson", "spearman"):
        cc = S.xcorr(SUBJ[s]["b_all"], method=m)
        for lr in ("0..+100", "-100..+100"):
            R, lag = peak(cc, lr)
            va.append({"rat": s, "corr": m, "lags": lr, "R": R, "lag_min": lag})
print("Variant with all stored events (raw delta, one correlogram per rat, all rats):")
print(pd.DataFrame(va).groupby(["corr", "lags"]).apply(summarise).round(3).to_string())
# the effect of the date repair, in the rats it was applied to
rep = [s for s in RATS if SUBJ[s]["swapped"]]
cmp_rows = []
for s in rep:
    st = SUBJ[s]["stored"]
    n_st = int((np.isfinite(st.rate) & np.isfinite(st.delta)).sum())
    cmp_rows.append({"rat": s, "bins_repaired": int((np.isfinite(SUBJ[s]["b"].rate) & np.isfinite(SUBJ[s]["b"].delta)).sum()),
                     "R10_repaired": CC[(s, "raw", "spearman", "rat")][LAG_MIN == 10][0],
                     "bins_as_stored": n_st, "R10_as_stored": S.xcorr(st, method="spearman")[LAG_MIN == 10][0] if n_st else np.nan})
print("Spearman R at +10 min, with the repaired and the stored glucose timestamps:")
print(pd.DataFrame(cmp_rows).round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s in RATS:
    ax[0].plot(LAG_MIN, CC[(s, "raw", "spearman", "rat")], color="0.7", lw=0.8)
med = np.nanmedian([CC[(s, "raw", "spearman", "rat")] for s in RATS], axis=0)
ax[0].plot(LAG_MIN, med, color="C3", lw=2)
ax[0].axvline(0, color="k", lw=0.5); ax[0].set_xlabel("offset (min; + = glucose after SPW-Rs)"); ax[0].set_ylabel("Spearman R")
ax[0].set_title("SPW-R rate vs delta current, per rat (grey) and median (red)")
allcc = np.array([CC[(s, "raw", "spearman", "rat")] for s in RATS])
ax[1].imshow(allcc, aspect="auto", cmap="RdBu_r", vmin=-0.4, vmax=0.4, extent=[-102.5, 102.5, len(RATS) - 0.5, -0.5])
ax[1].set_yticks(range(len(RATS))); ax[1].set_yticklabels(RATS, fontsize=7); ax[1].set_xlabel("offset (min)")
plt.tight_layout(); plt.show()
print(f"plotted R range {np.nanmin(allcc):.3f} to {np.nanmax(allcc):.3f}; colour scale -0.4 to 0.4, "
      f"{int((np.abs(allcc) > 0.4).sum())} values outside it; median correlogram minimum {np.nanmin(med):.3f} at {LAG_MIN[np.nanargmin(med)]} min")

**Result: reproduced, with 18 rats where the paper has 30, under 3 of the 16 combinations of
readings, resting on an assumption about the alignment.** With Pearson correlation, one correlogram per rat and lags searched over -100 to +100
min, the mean peak correlation is -0.176 (raw delta current) or -0.180 (high-pass) at a mean lag of
14.2 or 14.4 min, inside both tolerances; with Spearman correlation, one correlogram per session
and the high-pass delta, -0.184 at 8.8 min. The peak is negative in every rat. Mean R lies within
the tolerance under all 16 combinations (-0.159 to -0.216); the mean lag does not under the other
13 (15.8 to 33.1 min), and it is unstable: the lags of single rats spread with an s.d. of 26 to 42
min, and leaving out the rats with less than 6 h of data leaves 1 combination of 16 that matches.
The median correlogram has its minimum, -0.108, at +5 min. CGM1, the one rat labelled as the
ripple and glucose cohort, peaks at +5 min with R -0.101 (Pearson) or -0.194 (Spearman). The
outcome rests on pairing the SPW-R count and the glucose change of the same 5-min interval, an
assumption: counting SPW-Rs over the 5 min after each reading moves every lag by +5 min, and then
none of the 8 per-rat combinations matches (mean lags 19.2 to 37.8 min), while taking the change to
the next reading moves them by -5 min and 4 of 8 match (mean lags 9.4 to 10.8 min).

## fig2ef-correlation-beyond-shuffle

> The correlation between SPW-R rate and glucose change at a 10-min lag is more negative than the
> correlations obtained when SPW-R rate is circularly permuted.

Per rat, the correlation at +10 min against 100 circular shuffles of SPW-R rate within each run
(the claims file's shuffle), with the two-sample Kolmogorov-Smirnov test the claims file names.
Both correlation types are shown.

In [ ]:
rng = np.random.default_rng(20210811)
lag2 = int(np.where(S.LAGS == 2)[0][0])
obs, shuf = {"pearson": [], "spearman": []}, {"pearson": [], "spearman": []}
for s in RATS:
    b = SUBJ[s]["b"]
    for m in obs:
        obs[m].append(CC[(s, "raw", m, "rat")][lag2])
    for i in range(100):
        sb = S.circular_shuffle(b, rng)
        for m in shuf:
            shuf[m].append(S._corr(*S.lagged_pairs(sb, 2), m))
for m in obs:
    o, sh = np.array(obs[m]), np.array(shuf[m])
    ks = stats.ks_2samp(o, sh[np.isfinite(sh)])
    print(f"{m}: observed median {np.nanmedian(o):.3f} ({len(o)} rats, {np.sum(o < 0)} negative), shuffled median {np.nanmedian(sh):.4f} "
          f"(n = {np.isfinite(sh).sum()}, 2.5-97.5%: {np.nanpercentile(sh, 2.5):.3f} to {np.nanpercentile(sh, 97.5):.3f}); KS D = {ks.statistic:.3f}, p = {ks.pvalue:.2e}")
    per = np.array(sh).reshape(len(RATS), 100)
    print("   rats whose observed value is below all of their own 100 shuffles:", int(np.sum(o < np.nanmin(per, axis=1))))
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(np.array(shuf["spearman"]), bins=40, density=True, color="k", alpha=0.5, label="shuffled")
ax.hist(np.array(obs["spearman"]), bins=10, density=True, color="C3", alpha=0.7, label="observed")
ax.set_xlabel("Spearman R at +10 min"); ax.legend(); plt.show()

**Result: reproduced.** At +10 min the observed per-rat correlations have a median of -0.106
(Spearman; 16 of 18 rats negative) against 0.005 for 1,799 circular shuffles, KS p = 4.0e-06;
with Pearson, -0.076 against 0.002, KS p = 1.5e-04. Seven rats (Spearman) lie below all 100 of
their own shuffles.

## fig2g-clustered-ripples

> Isolated SPW-Rs have little or no effect on glucose, whereas clustered SPW-Rs (more than 30 per
> minute) are followed by a several-fold larger decrease.

As the Fig. 2g legend: the mean delta current 10 min after each 5-min epoch, grouped by the
number of SPW-Rs in the epoch, pooled over rats. Clustered is more than 150 SPW-Rs per epoch (a) or
more than 30 (b); isolated is 1 to 5 per epoch (claims file).

In [ ]:
ep = []
for s in RATS:
    b = SUBJ[s]["b"].copy()
    b["delta_10"] = b.groupby("run").delta.shift(-2)
    b["rat"] = s
    ep.append(b[np.isfinite(b.rate) & np.isfinite(b.delta_10)])
ep = pd.concat(ep)
print(f"{len(ep)} epochs from {ep.rat.nunique()} rats; SPW-Rs per epoch: median {ep['count'].median():.0f}, max {ep['count'].max()}")
iso = ep[(ep["count"] >= 1) & (ep["count"] <= 5)]
none = ep[ep["count"] == 0]
for name, thr in (("a: > 150 per epoch", 150), ("b: > 30 per epoch", 30)):
    cl = ep[ep["count"] > thr]
    t = stats.ttest_ind(cl.delta_10, iso.delta_10, equal_var=False)
    print(f"{name}: clustered n = {len(cl)} ({cl.rat.nunique()} rats), mean delta at +10 min {cl.delta_10.mean():.4f} nA; "
          f"isolated n = {len(iso)}, mean {iso.delta_10.mean():.4f} nA; ratio {cl.delta_10.mean() / iso.delta_10.mean():.2f}; "
          f"Welch t = {t.statistic:.2f}, p = {t.pvalue:.2g}")
print(f"epochs with no SPW-R: n = {len(none)}, mean {none.delta_10.mean():.4f} nA; all epochs mean {ep.delta_10.mean():.4f} nA")
bins = [-1, 0, 5, 15, 30, 60, 90, 120, 150, 200, 400]
ep["group"] = pd.cut(ep["count"], bins)
curve = ep.groupby("group", observed=True).delta_10.agg(["size", "mean", "sem"])
print(curve.round(4).to_string())
per_rat = ep.assign(cls=np.select([ep["count"].between(1, 5), ep["count"] > 150, ep["count"] > 30], ["isolated", "clustered_a", "clustered_b_only"], "other"))
print(per_rat.groupby(["rat", "cls"]).delta_10.mean().unstack().round(3).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(len(curve)); ax.errorbar(x, curve["mean"], curve["sem"], marker="o")
ax.set_xticks(x); ax.set_xticklabels([str(i) for i in curve.index], rotation=45, fontsize=7)
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("SPW-Rs per 5-min epoch"); ax.set_ylabel("delta current at +10 min (nA)"); plt.show()

**Result: reproduced under reading (a).** Epochs with more than 150 SPW-Rs are followed 10 min
later by a mean delta current of -0.191 nA (585 epochs, 13 rats), against +0.050 nA after epochs
with 1 to 5 SPW-Rs (1,718 epochs): negative and 3.9 times the magnitude, though the difference is
not significant by a Welch test (p = 0.059). Under reading (b), more than 30 per epoch, the mean is
-0.032 nA, smaller in magnitude than the isolated value, so (b) does not match. The curve over epoch
counts is not monotonic (the 120 to 150 group is +0.125 nA), and in 3 of the 13 rats with clustered
epochs their mean is not negative.

## edfig4-ripple-features

> Of the SPW-R features, the inter-SPW-R interval (clustering) most strongly modulates the glucose
> effect, then amplitude, then duration, and ripple frequency least.

For each SPW-R, the delta current of the reading 10 min after the 5-min interval holding it, as in
the Extended Data Fig. 4b legend; SPW-Rs pooled over rats and binned by the percentile of each
feature (deciles, a; 5-percentile bins, b). Features from the Methods: amplitude is the maximum
normalized squared signal (`peak_normed_power`), duration is stop minus start, frequency is
`peak_frequency`, and the inter-SPW-R interval is the time from the previous SPW-R in the same
file (the claims file does not say previous or next; previous is a standard choice). The
influence of a feature is the range of the bin means (claims file).

In [ ]:
evs = []
for s in RATS:
    d = SUBJ[s]
    e = d["ev"].copy()
    e["interval"] = e.groupby("file").t.diff()
    b = d["b"].copy()
    b["delta_10"] = b.groupby("run").delta.shift(-2)
    b = b[np.isfinite(b.rate)]
    # the reading whose 5-min interval holds the SPW-R: the first reading at or after it
    idx = np.searchsorted(b.t.values, e.t.values, side="left")
    ok = idx < len(b)
    idx = np.where(ok, idx, 0)
    inside = ok & (b.t.values[idx] - e.t.values < S.STEP)
    e["delta_10"] = np.where(inside, b.delta_10.values[idx], np.nan)
    e["rat"] = s
    evs.append(e[np.isfinite(e.delta_10)])
evs = pd.concat(evs)
evs["duration_ms"] = 1000 * evs.duration
print(f"{len(evs)} SPW-Rs from {evs.rat.nunique()} rats with a delta current 10 min later")
feats = {"interval": "interval", "amplitude": "peak_normed_power", "duration": "duration_ms", "frequency": "peak_frequency"}
res = []
for nb, label in ((10, "a: deciles"), (20, "b: 5-percentile bins")):
    for f, col in feats.items():
        x = evs[[col, "delta_10"]].dropna()
        q = pd.qcut(x[col].rank(method="first"), nb, labels=False)
        m = x.groupby(q).delta_10.mean()
        res.append({"bins": label, "feature": f, "n": len(x), "range_nA": m.max() - m.min(),
                    "lowest_bin_mean": m.iloc[0], "highest_bin_mean": m.iloc[-1], "min_bin": int(m.idxmin()), "max_bin": int(m.idxmax())})
R4 = pd.DataFrame(res)
print(R4.round(4).to_string())
for label, g in R4.groupby("bins"):
    print(label, "order by range:", " > ".join(g.sort_values("range_nA", ascending=False).feature))
# per rat, deciles
pr = []
for s, g in evs.groupby("rat"):
    row = {"rat": s, "n": len(g)}
    for f, col in feats.items():
        x = g[[col, "delta_10"]].dropna()
        m = x.groupby(pd.qcut(x[col].rank(method="first"), 10, labels=False)).delta_10.mean()
        row[f] = m.max() - m.min()
    pr.append(row)
pr = pd.DataFrame(pr)
pr["largest"] = pr[list(feats)].idxmax(axis=1)
print(pr.round(4).to_string()); print("largest range per rat:", pr.largest.value_counts().to_dict())

**Result: partially reproduced.** With deciles (a), the inter-SPW-R interval has the largest
range of the +10-min effect across bins (0.070 nA), then duration (0.069), frequency (0.053) and
amplitude (0.040), so the interval leads but the rest of the order differs. With 5-percentile bins
(b), duration (0.0967) and interval (0.0966) are tied ahead of amplitude (0.061) and frequency
(0.060). Per rat, amplitude has the largest range in 7 of 18 rats and the interval in 6.

## fig2h-phase-reset

> The phase dispersion of the glucose rhythm decreases after SPW-Rs, more so for higher SPW-R
> rates.

As the claims file: glucose phase is the Hilbert phase of delta current band-passed at periods of
15 to 75 min (a) or 15 to 90 min (b) (second-order Butterworth, zero phase, within each run of at
least 24 h, a standard choice); peaks are local maxima of SPW-R rate; peaks are pooled over rats
and grouped into deciles of peak rate; for each decile the circular s.d. of glucose phase is taken
at offsets -100 to +100 min; the decrease is the baseline (offset 0, a; mean over -100 to 0 min,
b) minus the minimum from 0 to +30 min.

In [ ]:
FS = 1 / S.STEP

def band_phase(b, lo_min, hi_min, col="delta", min_len=288):
    """Hilbert phase and envelope of `col` band-passed between periods lo_min and hi_min, per run."""
    ph = np.full(len(b), np.nan); env = np.full(len(b), np.nan)
    sos = signal.butter(2, [1 / (hi_min * 60), 1 / (lo_min * 60)], btype="band", fs=FS, output="sos")
    vals = b[col].values
    for _, idx in b.groupby("run").indices.items():
        x = vals[idx]
        if len(idx) < min_len:
            continue
        x = np.where(np.isfinite(x), x, np.nanmean(x))
        z = signal.hilbert(signal.sosfiltfilt(sos, x - x.mean()))
        ph[idx] = np.angle(z); env[idx] = np.abs(z)
    return ph, env

def circ_sd(a):
    a = a[np.isfinite(a)]
    if len(a) < 3:
        return np.nan
    return np.sqrt(-2 * np.log(np.abs(np.mean(np.exp(1j * a)))))

OFFS = np.arange(-20, 21)
for band in ((15, 75), (15, 90)):
    pk = []
    for s in RATS:
        b = SUBJ[s]["b"]
        ph, _ = band_phase(b, *band)
        r = b.rate.values
        for _, idx in b.groupby("run").indices.items():
            rr = np.where(np.isfinite(r[idx]), r[idx], -1)
            peaks, _ = signal.find_peaks(rr)
            for p in peaks:
                if not np.isfinite(r[idx][p]):
                    continue
                row = np.full(len(OFFS), np.nan)
                for j, o in enumerate(OFFS):
                    if 0 <= p + o < len(idx):
                        row[j] = ph[idx][p + o]
                if np.isfinite(row[OFFS == 0][0]):
                    pk.append((s, r[idx][p], row))
    rates = np.array([x[1] for x in pk]); P = np.array([x[2] for x in pk])
    dec = pd.qcut(pd.Series(rates).rank(method="first"), 10, labels=False).values
    tab = []
    for k in range(10):
        csd = np.array([circ_sd(P[dec == k, j]) for j in range(len(OFFS))])
        post = csd[(OFFS >= 0) & (OFFS <= 6)].min()
        tab.append({"decile": k + 1, "peaks": int((dec == k).sum()), "median_rate": np.median(rates[dec == k]),
                    "csd_0": csd[OFFS == 0][0], "csd_pre_mean": csd[OFFS <= 0].mean(), "min_0_30": post,
                    "decrease_a": csd[OFFS == 0][0] - post, "decrease_b": csd[OFFS <= 0].mean() - post})
    tab = pd.DataFrame(tab)
    print(f"band {band[0]}-{band[1]} min: {len(rates)} peaks of SPW-R rate from {len(set(x[0] for x in pk))} rats")
    print(tab.round(3).to_string())
    for col in ("decrease_a", "decrease_b"):
        rho = stats.spearmanr(tab[col], tab.median_rate)
        print(f"  {col}: top decile {tab[col].iloc[-1]:.3f}; Spearman with median rate rho = {rho.statistic:.3f}, p = {rho.pvalue:.3g}")

**Result: not reproduced.** Under neither band nor baseline is the decrease of phase dispersion
correlated with SPW-R rate at p < 0.05: Spearman rho -0.24 (p = 0.51) for baseline (a) and 0.55
(p = 0.098) for baseline (b), the same in both bands. The decrease in the top decile is positive
under all four readings (0.022 to 0.506), but it is positive in every decile, including the
lowest (0.23 to 0.47).

## edfig5-cross-frequency-coupling

> SPW-R rate and glucose show power-power and phase-power coupling at periods of 15 to 75 min,
> with the faster SPW-R rhythm (about 40 min) modulating the slower glucose rhythm (about 60 min).

As the claims file: band-pass at log-spaced periods (here 12 to 120 min: a band centred at 10 min
would sit at the Nyquist period of 5-min readings), Hilbert envelopes; power-power is the
correlation of the SPW-R-rate and glucose envelopes for each pair of periods per rat, against
circular shuffles of SPW-R rate (20 per rat, fewer than the paper's 100 to keep the run short; a
standard choice), with a paired t-test across rats on the mean over 15 to 60 min. Phase-power is
SPW-R rate (z-scored per rat) by glucose phase (the 15 to 75 min band), with a Rayleigh test of
the rate-weighted phases pooled over rats (weights: z-scored rate minus its minimum, so that they
are positive; a choice).

In [ ]:
rng = np.random.default_rng(5)
PER = np.round(np.geomspace(12, 120, 12), 1)
def envs(b, col):
    return np.array([band_phase(b, max(p / 1.25, 10.5), p * 1.25, col=col)[1] for p in PER])

def comod(eg, er):
    out = np.full((len(PER), len(PER)), np.nan)
    for i in range(len(PER)):
        for j in range(len(PER)):
            ok = np.isfinite(eg[i]) & np.isfinite(er[j])
            if ok.sum() > 50:
                out[i, j] = np.corrcoef(eg[i][ok], er[j][ok])[0, 1]
    return out

patch = (PER >= 15) & (PER <= 60)
obs_m, sh_m, rats5, mats = [], [], [], []
for s in RATS:
    b = SUBJ[s]["b"].copy()
    b["rate_f"] = b.rate.fillna(b.rate.mean())
    eg = envs(b, "delta"); er = envs(b, "rate_f")
    if np.all(np.isnan(eg)):
        continue
    M = comod(eg, er)
    sh = []
    for i in range(20):
        sb = S.circular_shuffle(b, rng, x="rate_f")
        sh.append(np.nanmean(comod(eg, envs(sb, "rate_f"))[np.ix_(patch, patch)]))
    rats5.append(s); mats.append(M); obs_m.append(np.nanmean(M[np.ix_(patch, patch)])); sh_m.append(np.mean(sh))
obs_m, sh_m = np.array(obs_m), np.array(sh_m)
t = stats.ttest_rel(obs_m, sh_m)
print(f"{len(rats5)} rats with a run of at least 24 h: {rats5}")
print(f"power-power, mean over 15-60 min: observed {obs_m.mean():.3f} (per rat {np.round(obs_m, 3)}), shuffled {sh_m.mean():.3f}; paired t = {t.statistic:.2f}, p = {t.pvalue:.2g}")
MM = np.nanmean(mats, axis=0)
i, j = np.unravel_index(np.nanargmax(MM), MM.shape)
print(f"mean comodulogram peak R = {MM[i, j]:.3f} at glucose period {PER[i]} min and SPW-R period {PER[j]} min")
off = np.nanmean([MM[a, c] for a in range(len(PER)) for c in range(len(PER)) if a > c])
below = np.nanmean([MM[a, c] for a in range(len(PER)) for c in range(len(PER)) if a < c])
print(f"mean R with glucose period longer than SPW-R period {off:.3f}; shorter {below:.3f}; diagonal {np.nanmean(np.diag(MM)):.3f}")
print(pd.DataFrame(MM, index=[f"glu {p}" for p in PER], columns=[f"spw {p}" for p in PER]).round(2).to_string())
# phase-power
ph_all, w_all, z_all = [], [], []
for s in rats5:
    b = SUBJ[s]["b"]
    ph, _ = band_phase(b, 15, 75)
    z = (b.rate - b.rate.mean()) / b.rate.std()
    ok = np.isfinite(ph) & np.isfinite(z)
    ph_all.append(ph[ok]); z_all.append(z.values[ok])
ph_all = np.concatenate(ph_all); z_all = np.concatenate(z_all); w = z_all - z_all.min()
Rw = np.abs(np.sum(w * np.exp(1j * ph_all))) / w.sum()
n = len(ph_all); Z = n * Rw ** 2
p_ray = np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - (n * Rw) ** 2)) - (1 + 2 * n))
mean_phase = np.degrees(np.angle(np.sum(w * np.exp(1j * ph_all))))
print(f"phase-power: {n} bins; weighted resultant length {Rw:.4f}, Rayleigh Z = {Z:.1f}, p = {p_ray:.3g}, preferred glucose phase {mean_phase:.0f} deg")
pb = pd.cut(ph_all, np.linspace(-np.pi, np.pi, 13))
print("mean z-scored SPW-R rate by glucose phase:", pd.Series(z_all).groupby(pb, observed=True).mean().round(3).values)

**Result: partially reproduced.** Power-power coupling over 15 to 60 min exceeds the shuffles
(mean R 0.107 against -0.007, paired t = 5.07, p = 9.5e-05, 18 rats) and the rate-weighted
glucose phases are non-uniform (Rayleigh p = 1.4e-13, resultant length 0.038). The comodulogram
peak (R 0.180) lies on the diagonal, at 51.9 min for both signals, and the mean correlation with
the glucose period longer than the SPW-R period equals that with it shorter (0.082 each), so the
shift off the diagonal is not found.

## edfig1-glucose-rhythm

> Interstitial glucose fluctuations show an approximately 60-min oscillation, with power elevated
> above a linear best fit to the spectrum at periods of 15 to 90 min.

As Extended Data Fig. 1g-i: power spectral density of delta current per rat (Welch, 24-h segments
within runs, a standard choice), a line fitted on log-log axes (the claims file's standard
choice) and subtracted, averaged across rats; and the mean autocorrelogram of delta current. All
subjects with at least one 24-h run of readings are used (the glucose series do not need the
electrophysiology).

In [ ]:
psd_rows, ac_rows, used = [], [], []
for s in sorted(SUBJ):
    glu = SUBJ[s]["glu"].copy()
    glu["delta"] = np.where(glu.run.eq(glu.run.shift(1)), glu.nA.diff(), np.nan)
    segs = [g.delta.values[1:] for _, g in glu.groupby("run") if len(g) >= 289]
    if not segs:
        continue
    ps = []
    for x in segs:
        f, p = signal.welch(np.nan_to_num(x), fs=FS, nperseg=288)
        ps.append(p * len(x))
    psd_rows.append(np.sum(ps, axis=0) / sum(len(x) for x in segs))
    ac = []
    for x in segs:
        x = np.nan_to_num(x - np.nanmean(x))
        c = np.correlate(x, x, "full")[len(x) - 1:len(x) - 1 + 37] / np.dot(x, x)
        ac.append(c)
    ac_rows.append(np.mean(ac, axis=0)); used.append(s)
F1 = f[1:]; period = 1 / F1 / 60
resid = []
for p in psd_rows:
    lp = np.log10(p[1:]); coef = np.polyfit(np.log10(F1), lp, 1)
    resid.append(lp - np.polyval(coef, np.log10(F1)))
resid = np.array(resid); mres = resid.mean(axis=0)
print(f"{len(used)} subjects: {used}")
tbl = pd.DataFrame({"period_min": period, "mean_residual_log10": mres, "rats_positive": (resid > 0).sum(axis=0)})
print(tbl[(tbl.period_min >= 10) & (tbl.period_min <= 300)].round(3).to_string())
band = (period >= 15) & (period <= 90)
print(f"residual over 15-90 min: mean {mres[band].mean():.3f}, positive at {np.sum(mres[band] > 0)} of {band.sum()} frequencies")
ipk = np.argmax(np.where((period >= 10) & (period <= 300), mres, -np.inf))
print(f"peak of the mean residual at {period[ipk]:.1f} min")
mac = np.mean(ac_rows, axis=0); lags_min = np.arange(37) * 5
pk, _ = signal.find_peaks(mac)
print("mean autocorrelogram:", dict(zip(lags_min[:25], np.round(mac[:25], 3))))
print("first side peak of the mean autocorrelogram at", lags_min[pk[0]] if len(pk) else None, "min",
      f"(value {mac[pk[0]]:.3f})" if len(pk) else "")
per_rat_pk = [period[np.argmax(np.where((period >= 10) & (period <= 300), r, -np.inf))] for r in resid]
print("per subject peak of residual (min):", dict(zip(used, np.round(per_rat_pk, 1))))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].semilogx(period, mres); ax[0].axhline(0, color="k", lw=0.5); ax[0].set_xlabel("period (min)"); ax[0].set_ylabel("log10 power - fit")
ax[1].plot(lags_min, mac); ax[1].set_xlabel("lag (min)"); plt.show()

**Result: partially reproduced.** In 25 subjects the power of delta current above a log-log
linear fit is elevated from about 22 to 131 min, with its peak at 51.4 min (per subject 32 to 60
min), outside 60 +- 6 min; below 22 min the residual is negative, so it is positive at only 51 of
the 81 frequencies between 15 and 90 min. The mean autocorrelogram has its first side peak at 70
min with a value of 0.013, outside the tolerance.

## edfig1-glucose-circadian-meals

> Glucose concentrations are modulated over the circadian cycle and rise after meals.

The meal times of the fasting experiment are not in the dandiset, so the meal part and its
expected values (36 meals, 4 rats) cannot be tested. The circadian part of the criterion is run:
current by hour of day averaged per subject, one-way ANOVA of hour on the hourly means pooled over
subjects.

In [ ]:
hourly = []
for s in sorted(SUBJ):
    glu = SUBJ[s]["glu"]
    h = S.clock(SUBJ[s]["ref"], glu.t).hour
    m = glu.nA.groupby(h).mean()
    if len(m) == 24:
        hourly.append(pd.DataFrame({"subject": s, "hour": m.index, "nA": m.values}))
hourly = pd.concat(hourly)
an = stats.f_oneway(*[g.nA.values for _, g in hourly.groupby("hour")])
print(f"{hourly.subject.nunique()} subjects; one-way ANOVA of hour of day: F = {an.statistic:.2f}, p = {an.pvalue:.3g}")
print("mean current by hour (nA):", hourly.groupby("hour").nA.mean().round(1).to_dict())

**Result: blocked (meal times are not in the dandiset).** The circadian part of the criterion
does not find a modulation: one-way ANOVA of hour of day on the hourly means of 25 subjects,
F = 0.46, p = 0.99; the mean current ranges from 28.6 nA (06:00 to 07:00) to 31.3 nA (20:00).

## edfig1-sensor-decay

> The raw sensor current decays slowly over days after implantation, while the magnitude of its
> fluctuations (delta current) stays stable.

Per placement, daily mean current and daily mean absolute delta current against day since the
first reading (claims file). A placement is a run of consecutive readings lasting at least 3 days
(a standard choice); slopes are compared as fractions of the first-day value.

In [ ]:
rows = []
for s in sorted(SUBJ):
    glu = SUBJ[s]["glu"]
    for r, g in glu.groupby("run"):
        if len(g) < 3 * 288:
            continue
        day = ((g.t - g.t.iloc[0]) // 86400).astype(int)
        ad = g.nA.diff().abs()
        dm = g.nA.groupby(day).mean(); da = ad.groupby(day).mean()
        full = g.groupby(day).size() >= 144
        dm, da = dm[full], da[full]
        sl_c = np.polyfit(dm.index, dm.values, 1)[0]; sl_d = np.polyfit(da.index, da.values, 1)[0]
        rows.append({"subject": s, "run": r, "days": len(dm), "first_day_nA": dm.iloc[0], "slope_nA_per_day": sl_c,
                     "rel_slope_current": sl_c / dm.iloc[0], "first_day_abs_delta": da.iloc[0], "rel_slope_abs_delta": sl_d / da.iloc[0]})
dec = pd.DataFrame(rows)
print(dec.round(3).to_string())
print(f"placements: {len(dec)} from {dec.subject.nunique()} subjects; current slope negative in {int((dec.slope_nA_per_day < 0).sum())}; "
      f"|relative slope of |delta|| smaller than that of the current in {int((dec.rel_slope_abs_delta.abs() < dec.rel_slope_current.abs()).sum())}")
print(f"median relative slope per day: current {dec.rel_slope_current.median():.3f}, |delta| {dec.rel_slope_abs_delta.median():.3f}")

**Result: reproduced.** The daily mean current falls in all 31 placements (median -10.7% of
the first-day value per day), and the daily mean absolute delta current changes less in relative
terms (median -4.7% per day; smaller in magnitude than the current's in 18 of 31 placements).

## supp-ripple-rate

> CA1 SPW-Rs occur at 9.5 (+- 8.2) per minute on average, about 14,000 per day, ranging from none
> for several minutes to more than 40 in a minute.

SPW-R count in 1-min bins over all CA1 recording (minutes fully covered by a file with a CA1
SPW-R table). Readings for the rats: (a) every rat with CA1 SPW-Rs and glucose together, (b) the
ripple and glucose cohort, of which only CGM1 is labelled in the files. All subjects with a CA1
table are also shown.

In [ ]:
def minute_counts(d):
    out = []
    for x, y in d["iv"]:
        edges = np.arange(np.ceil(x / 60) * 60, y + 1e-9, 60)
        if len(edges) < 2:
            continue
        e = d["ev"].t.values
        out.append(np.diff(np.searchsorted(e, edges)))
    return np.concatenate(out) if out else np.array([], int)

MC = {s: minute_counts(SUBJ[s]) for s in SUBJ if SUBJ[s]["ev"] is not None}
rows = [{"subject": s, "minutes": len(c), "mean_per_min": c.mean(), "sd_per_min": c.std(), "max_per_min": c.max(),
         "per_day": c.mean() * 1440, "zero_runs_ge_5min": int(np.sum(np.diff(np.flatnonzero(np.diff(np.r_[1, (c == 0).astype(int), 1]))) [::2] >= 5))}
        for s, c in MC.items()]
mr = pd.DataFrame(rows)
print(mr.round(2).to_string())
for name, keep in (("a: rats with SPW-Rs and glucose", RATS), ("b: CGM1", ["CGM1"]), ("all subjects with a CA1 table", list(MC))):
    c = np.concatenate([MC[s] for s in keep])
    print(f"{name}: {len(keep)} rats, {len(c) / 60:.0f} h; mean {c.mean():.2f} (sd {c.std():.2f}) per minute, {c.mean() * 1440:.0f} per day, "
          f"max {c.max()} per minute; mean of rat means {mr[mr.subject.isin(keep)].mean_per_min.mean():.2f}")
    c5 = np.concatenate([minute_counts({"iv": SUBJ[s]["iv"], "ev": SUBJ[s]["ev_all"]}) for s in keep])
    print(f"   variant, all stored events (no 5 s.d. threshold): mean {c5.mean():.2f} per minute, {c5.mean() * 1440:.0f} per day, max {c5.max()}")
print("minutes with more than 40 SPW-Rs:", {s: int((c > 40).sum()) for s, c in MC.items()})

**Result: reproduced.** Over the 18 rats with SPW-Rs and glucose (2,264 h), SPW-Rs with a peak of
at least 5 s.d. occur at 10.19 (s.d. 11.15) per minute, 14,671 per day, inside both tolerances, and
up to 258 in a minute; minutes with more than 40 occur in 18 of 22 subjects. With all stored events
the mean is 11.07 per minute and 15,941 per day, outside the tolerances; CGM1 alone gives 10.76 per
minute and 15,494 per day.

## supp-ripples-stable-across-days

> In each rat the shape, amplitude and rate of SPW-Rs are consistent across days.

Per rat and calendar day (days with at least 6 h of CA1 recording, a standard choice), the median
amplitude (`peak_normed_power`, the Methods' maximum normalized squared signal; `peak_amplitude`
shown as a variant), median duration and mean rate; Spearman correlation with day, and whether
every daily value stays within 20% of the rat's overall median (claims file).

In [ ]:
rows = []
for s, d in SUBJ.items():
    if d["ev"] is None:
        continue
    e = d["ev"].copy(); e["day"] = S.clock(d["ref"], e.t).normalize()
    secs = pd.Series(np.concatenate([np.arange(x, y, 60) for x, y in d["iv"]]))
    mins_per_day = S.clock(d["ref"], secs).normalize().value_counts()
    good = mins_per_day[mins_per_day >= 360].index
    g = e[e.day.isin(good)].groupby("day")
    daily = pd.DataFrame({"amp": g.peak_normed_power.median(), "amp_uV": g.peak_amplitude.median(),
                          "dur": g.duration.median(), "rate": g.size() / mins_per_day[good].reindex(g.size().index)})
    if len(daily) < 3:
        continue
    row = {"subject": s, "days": len(daily)}
    for k in daily:
        rho = stats.spearmanr(np.arange(len(daily)), daily[k])
        med = e[k if k not in ("amp", "amp_uV", "dur") else {"amp": "peak_normed_power", "amp_uV": "peak_amplitude", "dur": "duration"}[k]].median() if k != "rate" else daily[k].median()
        row[f"{k}_rho"] = rho.statistic; row[f"{k}_p"] = rho.pvalue
        row[f"{k}_within20"] = bool(np.all(np.abs(daily[k] / med - 1) <= 0.2))
        row[f"{k}_maxdev"] = float(np.max(np.abs(daily[k] / med - 1)))
    rows.append(row)
st = pd.DataFrame(rows)
print(st.round(3).to_string())
for k in ("amp", "amp_uV", "dur", "rate"):
    ok = (st[f"{k}_p"] >= 0.05) | st[f"{k}_within20"]
    print(f"{k}: consistent (Spearman p >= 0.05 or within 20%) in {int(ok.sum())} of {len(st)} rats; "
          f"within 20% in {int(st[f'{k}_within20'].sum())}; Spearman p < 0.05 in {int((st[f'{k}_p'] < 0.05).sum())}")

**Result: reproduced.** Daily median amplitude (peak normed power) and duration show no trend
with day (no Spearman p < 0.05) in all 15 rats with at least 3 days, and the daily rate in 11 of 15
(13 counting rats whose rate stays within 20%). Amplitude stays within 20% of the median in only 5
rats, and the rate in 4, so the consistency rests mostly on the absence of trends. Amplitude in
microvolts trends with day in 5 rats.

## edfig3-ripple-rate-rhythms

> SPW-R rate varies with circadian and ultradian rhythms, with spectral peaks at about 24 h and
> about 4 h, in 1,568.5 recording hours from 8 rats.

Power spectral density of SPW-R rate in 5-min bins per rat, averaged over rats (claims file). The
recordings have gaps, so the spectrum is a Lomb-Scargle periodogram of the covered bins (a
standard choice for unevenly sampled data). The 8 rats with the most CA1 recording are used, and
all rats with at least 48 h as a variant.

In [ ]:
per_h = np.linspace(1, 48, 400)
freqs = 2 * np.pi / (per_h * 3600)
LS, hrs = {}, {}
for s, d in SUBJ.items():
    if d["ev"] is None:
        continue
    t_all, r_all = [], []
    for x, y in d["iv"]:
        edges = np.arange(x, y - 299, 300)
        if len(edges) < 2:
            continue
        c = np.diff(np.searchsorted(d["ev"].t.values, np.r_[edges, edges[-1] + 300]))
        t_all.append(edges + 150); r_all.append(c / 5)
    t_all = np.concatenate(t_all); r_all = np.concatenate(r_all)
    hrs[s] = len(t_all) * 5 / 60
    if hrs[s] >= 48:
        p = signal.lombscargle(t_all, r_all - r_all.mean(), freqs, normalize=True)
        LS[s] = p
top8 = sorted(hrs, key=hrs.get, reverse=True)[:8]
print("CA1 recording hours per subject:", {k: round(v, 1) for k, v in sorted(hrs.items(), key=lambda x: -x[1])})
for name, keep in (("8 rats with the most recording", top8), ("all rats with >= 48 h", list(LS))):
    m = np.mean([LS[s] for s in keep], axis=0)
    pk, _ = signal.find_peaks(m)
    order = pk[np.argsort(m[pk])[::-1]]
    print(f"{name}: {len(keep)} rats, {sum(hrs[s] for s in keep):.1f} h; local peaks (period h, power): "
          f"{[(round(per_h[i], 1), round(m[i], 4)) for i in order[:8]]}")
    print(f"   peak nearest 24 h: {per_h[pk][np.argmin(np.abs(per_h[pk] - 24))]:.1f} h; nearest 4 h: {per_h[pk][np.argmin(np.abs(per_h[pk] - 4))]:.1f} h")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(per_h, np.mean([LS[s] for s in top8], axis=0)); ax.set_xlabel("period (h)"); ax.set_ylabel("normalized power"); plt.show()

**Result: partially reproduced.** The mean Lomb-Scargle spectrum of the 8 rats with the most
recording has local peaks at 24.0 h and 3.8 h, both inside the tolerances, but they are not its
largest peaks (42.3 and 19.5 h are larger), and the 8 rats hold 1,889.8 h of recording, outside
1,568.5 +- 156.9 h.

## edfig2-refractory-period

> Inter-ripple intervals drop to zero below about 20 ms, a refractory period between ripples, with
> peaks reflecting bursts of ripples.

Histogram of intervals between consecutive SPW-R peaks within a file, per rat, 1-ms bins (claims
file); the shortest interval with counts.

In [ ]:
rows = []
for s, d in SUBJ.items():
    if d["ev"] is None:
        continue
    iri = d["ev"].groupby("file").t.diff().dropna().values * 1000
    h, _ = np.histogram(iri, bins=np.arange(0, 501))
    nz = np.flatnonzero(h)
    gap = d["ev"].t.values[1:] - d["ev"].t.values[:-1]
    rows.append({"subject": s, "intervals": len(iri), "shortest_ms": iri.min(), "first_bin_with_counts_ms": nz[0] if len(nz) else np.nan,
                 "count_below_20ms": int((iri < 20).sum()), "count_20_30ms": int(((iri >= 20) & (iri < 30)).sum()),
                 "histogram_peak_ms": int(np.argmax(h)), "within_18_22": 18 <= iri.min() <= 22})
rf = pd.DataFrame(rows)
print(rf.round(2).to_string())
print(f"shortest interval within 18-22 ms in {int(rf.within_18_22.sum())} of {len(rf)} rats; median shortest {rf.shortest_ms.median():.1f} ms")

**Result: not reproduced.** The shortest interval between SPW-R peaks is 25.6 to 38.4 ms (median
34.4 ms) and no interval below 20 ms occurs in any of 22 rats; no rat has its shortest interval in
18 to 22 ms. The stored tables appear to merge or drop events closer than about 25 to 30 ms.

## edfig4-nrem-wake

> The SPW-R and glucose cross-correlogram does not differ between NREM sleep and waking, and its
> negative peak does not change across state transitions.

Sleep states are stored for six subjects; three of them (CGM1, CGM2, CGM3) have SPW-Rs and glucose
together. As the claims file: blocks of 4 h; within each, the correlation at +10 min (Spearman, the
correlation type under which the main claim matched) between the SPW-R rate of bins mostly in NREM
(or mostly awake) and the delta current 10 min later; one value per block and state, unpaired
two-sided t-test (per-rat means, paired, as a variant). Transitions: for each change from an awake
to a NREM episode (and the reverse) where both episodes last at least 10 min (a choice), the most
negative correlation from +5 to +20 min in the 2 h before and the 2 h after, paired t-test.

In [ ]:
def state_of_bins(b, ss):
    lab = np.full(len(b), "", dtype=object)
    for state in ("Non-REM", "Awake"):
        g = ss[ss.label == state]
        cov_ = S.coverage(list(zip(g.start, g.stop)), b.t.values - S.STEP, b.t.values)
        lab[cov_ > 0.5] = state
    return lab

blocks, trans = [], []
for s in RATS:
    fs = SUBJ[s]["files"]
    if not any(f["sleep"] is not None and len(f["sleep"]) for f in fs):
        continue
    ss = pd.concat([f["sleep"] + [S.clock_offset(f, SUBJ[s]["ref"]), S.clock_offset(f, SUBJ[s]["ref"]), ""]
                    for f in fs if f["sleep"] is not None]).sort_values("start").reset_index(drop=True)
    b = SUBJ[s]["b"].copy()
    b["state"] = state_of_bins(b, ss)
    b["block"] = ((b.t - b.t.min()) // (4 * 3600)).astype(int)
    for blk, g in b.groupby("block"):
        for state in ("Non-REM", "Awake"):
            gg = g.copy(); gg.loc[gg.state != state, "rate"] = np.nan
            x, y = S.lagged_pairs(gg, 2)
            if len(x) >= 10:
                blocks.append({"rat": s, "block": blk, "state": state, "pairs": len(x), "R10": S._corr(x, y, "spearman")})
    # transitions between episodes of at least 10 min
    ss2 = ss[ss.label.isin(["Non-REM", "Awake"])].reset_index(drop=True)
    for i in range(1, len(ss2)):
        a, c = ss2.iloc[i - 1], ss2.iloc[i]
        if a.label == c.label or a.stop - a.start < 600 or c.stop - c.start < 600 or abs(c.start - a.stop) > 60:
            continue
        t0 = c.start
        vals = {}
        for side, (lo, hi) in (("before", (t0 - 7200, t0)), ("after", (t0, t0 + 7200))):
            g = b[(b.t > lo) & (b.t <= hi)]
            cc = [S._corr(*S.lagged_pairs(g, k), "spearman") for k in (1, 2, 3, 4)]
            vals[side] = np.nanmin(cc) if np.any(np.isfinite(cc)) else np.nan
        trans.append({"rat": s, "type": f"{a.label}->{c.label}", **vals})
blocks = pd.DataFrame(blocks); trans = pd.DataFrame(trans)
nrem = blocks[blocks.state == "Non-REM"].R10; wake = blocks[blocks.state == "Awake"].R10
nrem, wake = nrem.dropna(), wake.dropna()
t = stats.ttest_ind(nrem, wake)
print(f"blocks: NREM n = {len(nrem)}, mean R10 {nrem.mean():.3f}; awake n = {len(wake)}, mean {wake.mean():.3f}; unpaired t = {t.statistic:.2f}, p = {t.pvalue:.3g}")
pm = blocks.groupby(["rat", "state"]).R10.mean().unstack()
print(pm.round(3).to_string())
if len(pm) > 1:
    tp = stats.ttest_rel(pm["Non-REM"], pm["Awake"], nan_policy="omit"); print(f"per-rat means, paired: t = {tp.statistic:.2f}, p = {tp.pvalue:.3g}")
for typ, g in trans.dropna().groupby("type"):
    tt = stats.ttest_rel(g.before, g.after)
    print(f"{typ}: {len(g)} transitions from {g.rat.nunique()} rats; most negative R (+5 to +20 min) before {g.before.mean():.3f}, after {g.after.mean():.3f}; paired t = {tt.statistic:.2f}, p = {tt.pvalue:.3g}")

**Result: reproduced, on 3 rats where the paper has 8.** NREM and awake blocks do not differ at
p < 0.05 (mean R at +10 min -0.017 against -0.083, unpaired t p = 0.057; per-rat means paired
p = 0.27), and the most negative correlation does not change across awake-to-NREM (p = 0.15, 13
transitions) or NREM-to-awake transitions (p = 0.23, 23 transitions from one rat). The awake blocks
are more negative, near the threshold, and the test has little power with 3 rats.

## edfig4-no-hour-differs

> No hour of the day has a significantly different SPW-R and glucose correlation than any other
> hour (smallest P = 0.53).

As the claims file: per rat, cross-correlograms in 4-h windows of the day centred on each hour
(all days pooled), integrated over offsets +5 to +20 min (the sum of R at +5, +10, +15, +20 min;
Spearman, as for the main claim); Student's t-test across rats for every pair of hours. Rats with
at least 24 h of both signals (a standard choice, so that each hour has data).

In [ ]:
rats24 = list(cov[cov.hours_with_both >= 24].subject)
H = np.full((len(rats24), 24), np.nan)
for i, s in enumerate(rats24):
    b = SUBJ[s]["b"]
    hod = (b.clock.dt.hour + b.clock.dt.minute / 60).values
    for hcent in range(24):
        dist = np.abs(((hod - hcent) + 12) % 24 - 12)
        g = b.copy(); g.loc[dist > 2, "rate"] = np.nan
        H[i, hcent] = np.nansum([S._corr(*S.lagged_pairs(g, k), "spearman") for k in (1, 2, 3, 4)])
ps = []
for a in range(24):
    for c in range(a + 1, 24):
        ps.append((a, c, stats.ttest_ind(H[:, a], H[:, c]).pvalue, stats.ttest_rel(H[:, a], H[:, c]).pvalue))
ps = pd.DataFrame(ps, columns=["hour_a", "hour_b", "p_student", "p_paired"])
print(f"{len(rats24)} rats: {rats24}")
print("integrated R (+5 to +20 min) by hour, mean over rats:", np.round(np.nanmean(H, axis=0), 3))
print(f"{len(ps)} pairs of hours; Student's t-test: smallest p = {ps.p_student.min():.3f}, pairs with p < 0.05: {int((ps.p_student < 0.05).sum())}; "
      f"paired variant: smallest p = {ps.p_paired.min():.3g}, pairs with p < 0.05: {int((ps.p_paired < 0.05).sum())}")

**Result: not reproduced.** Student's t-test across 12 rats finds 10 of 276 pairs of hours at
p < 0.05 (smallest p = 0.010), where the paper reports a smallest P of 0.53; the paired variant
finds 25. The integrated correlation is most negative from 17:00 to 21:00 (-0.37 to -0.51) and
near zero from 02:00 to 05:00 (-0.04 to 0.05). Ten pairs is fewer than the 14 expected by chance
among 276 uncorrected tests.

## edfig9-dorsal-ventral

> Dorsal and ventral CA1 SPW-Rs are only weakly correlated, and ventral SPW-Rs are significantly
> less correlated with glucose fluctuations than dorsal SPW-Rs, which match the earlier dorsal
> cohorts.

The dorsal and ventral rats are the three subjects with `Dorsalripples` and `Ventralripples`
tables (CGM37, CGM48, Vanessa). The correlation of dorsal and ventral 5-min SPW-R rates is taken
over the files holding both tables. The glucose comparison needs glucose recorded with the
electrophysiology, which only Vanessa has; per-day correlations at +10 min (Spearman) for dorsal
and ventral SPW-Rs, paired (a) and unpaired (b) t-tests, and the dorsal values against the other
rats' per-rat values (unpaired t-test).

In [ ]:
rows = []
for s in ("CGM37", "CGM48", "Vanessa"):
    for f in SUBJ[s]["files"]:
        if "Dorsalripples" in f["tables"] and "Ventralripples" in f["tables"]:
            dur = f["lfp_n"] / f["lfp_rate"]
            edges = np.arange(0, dur + 1e-9, 300)
            if len(edges) < 3:
                continue
            dc = np.histogram(f["tables"]["Dorsalripples"].peak, edges)[0]; vc = np.histogram(f["tables"]["Ventralripples"].peak, edges)[0]
            rows.append({"subject": s, "file": f["path"].split("/")[1][-30:], "bins": len(dc), "dorsal": int(dc.sum()), "ventral": int(vc.sum()),
                         "r_5min": np.corrcoef(dc, vc)[0, 1] if dc.std() and vc.std() else np.nan, "_d": dc, "_v": vc})
dv = pd.DataFrame(rows)
print(dv.drop(columns=["_d", "_v"]).round(3).to_string())
for s, g in dv.groupby("subject"):
    d_, v_ = np.concatenate(list(g._d)), np.concatenate(list(g._v))
    print(f"{s}: pooled 5-min rate correlation dorsal vs ventral r = {np.corrcoef(d_, v_)[0, 1]:.3f} over {len(d_)} bins")
d = SUBJ["Vanessa"]
res = {}
for name in ("Dorsalripples", "Ventralripples"):
    ev, iv = S.subject_events(d["files"], tables=(name,), reference=d["ref"])
    bb = S.binned(d["glu"], ev, iv); bb["day"] = S.clock(d["ref"], bb.t).normalize()
    res[name] = {day: S._corr(*S.lagged_pairs(g, 2), "spearman") for day, g in bb.groupby("day") if (np.isfinite(g.rate) & np.isfinite(g.delta)).sum() >= 48}
pdays = pd.DataFrame(res)
print("Vanessa, per-day Spearman R at +10 min (days with at least 4 h of both):"); print(pdays.round(3).to_string())
pdays = pdays.dropna()
if len(pdays) >= 2:
    print("paired t:", stats.ttest_rel(pdays.Dorsalripples, pdays.Ventralripples), " unpaired t:", stats.ttest_ind(pdays.Dorsalripples, pdays.Ventralripples))
others = [CC[(s, "raw", "spearman", "rat")][LAG_MIN == 10][0] for s in RATS if s != "Vanessa"]
print(f"other rats' R at +10 min: mean {np.mean(others):.3f} (n = {len(others)}); Vanessa dorsal per-day values {np.round(pdays.Dorsalripples.values, 3)}; "
      f"unpaired t = {stats.ttest_ind(pdays.Dorsalripples, others).statistic:.2f}, p = {stats.ttest_ind(pdays.Dorsalripples, others).pvalue:.3g}")

**Result: blocked (other: one rat with two days).** Dorsal and ventral 5-min SPW-R rates are
correlated with r = 0.13 (Vanessa), 0.44 (CGM37) and 0.83 (CGM48), below 0.5 in two of three rats.
Only Vanessa has glucose with the electrophysiology, on two days: dorsal 0.115 and 0.239, ventral
-0.088 and 0.096 at +10 min, the ventral values more negative (paired p = 0.11), and Vanessa's dorsal
values differ from the other rats' (p = 0.009). Vanessa's sensor reads a median of 0.25 nA, so its
glucose series is doubtful.

## Size of the predictor models (edfig6-ripple-rate-best-predictor, edfig6d-pairwise-predictors, edfig6ef-latent-variables)

Six of the seven predictors of the Methods (power spectrum slope, theta residual, theta/delta
ratio, high-pass coherence between shanks, movement, and NREM or wake where no sleep states are
stored) have to be computed from the LFP and the accelerometer over every hour with glucose. The
LFP is stored in chunks of 707 samples by 32 channels (gzip), so even one channel means reading all
32. The cell below times the reading of 30 min of LFP and of accelerometer from one file and
scales it to the hours the models need.

In [ ]:
a = assets[assets.path == "sub-CGM31/sub-CGM31_ses-CGM31-720um-210209-085741_ecephys.nwb"].iloc[0]
with S.open_h5(a.asset_id) as h:
    lfp = h["processing/ecephys/LFP/ElectricalSeriesLFP/data"]; acc = h["acquisition/Accelerometer/data"]
    print("LFP chunks", lfp.chunks, lfp.compression, "; accelerometer chunks", acc.chunks, acc.compression,
          f"at {h['acquisition/Accelerometer/starting_time'].attrs['rate']} Hz")
    t0 = time.time(); x = lfp[: int(1250 * 1800), :]; t_lfp = time.time() - t0
    t0 = time.time(); y = acc[: int(5000 * 1800), :]; t_acc = time.time() - t0
hours = cov.hours_with_both.sum()
print(f"30 min of LFP (32 channels) read in {t_lfp:.1f} s, of accelerometer in {t_acc:.1f} s")
print(f"for the {hours:.0f} h with SPW-Rs and glucose: LFP {hours * 2 * t_lfp / 3600:.1f} h, accelerometer {hours * 2 * t_acc / 3600:.1f} h "
      f"of reading, against a notebook limit of 0.5 h")

## Blocked claims

- **edfig3e-ca1-firing-decrease, edfig8-lateral-septum-rectified.** No file holds spike units
  (data check). The lateral septum recordings of ref. 26 are on the lab's server, which is not a
  pinned public source.
- **edfig4-fasting-fed, edfig1-glucose-circadian-meals (meal part).** Meal times are not in the
  dandiset.
- **edfig6-ripple-rate-best-predictor, edfig6d-pairwise-predictors, edfig6ef-latent-variables.**
  The predictors have to be computed from the LFP and the accelerometer over every hour with
  glucose; the timing above puts this at about 9 h of reading.
- **fig3d-induced-ripples-lower-glucose, fig3-induced-effect-size, fig3ef-stimulation-dose-response,
  fig3f-stimulation-vs-shuffle, edfig7ab-stimulation-protocol, edfig7jm-induced-ripples-decoupled,
  edfig7-ppc-stimulation-no-effect.** No file holds stimulation times or a stimulation channel, and
  no subject is labelled as optogenetic or PPC.
- **fig4-cno-controls-no-change, fig4-ls-dreadd-decouples, fig4-ls-dreadd-chance-level,
  fig4e-dreadd-time-course, fig4c-medial-septum-no-effect, edfig10-expression-correlates-effect.**
  The recordings of the DREADD rats may be among the files, but nothing says which subjects had
  hM4Di in the lateral or medial septum or no virus, which days were CNO or vehicle days, or when
  the injections were given. The histology scores of Extended Data Fig. 10e are printed only in the
  figure.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig2-ripple-rate-predicts-glucose | reproduced under 3 of 16 combinations of readings (18 rats), resting on an assumption |
| fig2ef-correlation-beyond-shuffle | reproduced |
| fig2g-clustered-ripples | reproduced under reading (a) |
| edfig4-ripple-features | partially reproduced |
| fig2h-phase-reset | not reproduced |
| edfig5-cross-frequency-coupling | partially reproduced |
| edfig1-glucose-rhythm | partially reproduced |
| edfig1-glucose-circadian-meals | blocked: data missing |
| edfig1-sensor-decay | reproduced |
| supp-ripple-rate | reproduced |
| supp-ripples-stable-across-days | reproduced |
| edfig3-ripple-rate-rhythms | partially reproduced |
| edfig2-refractory-period | not reproduced |
| edfig4-nrem-wake | reproduced (3 rats) |
| edfig4-no-hour-differs | not reproduced |
| edfig9-dorsal-ventral | blocked: other (one rat) |
| 19 other claims | blocked (see above) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: 18 rats in place of 30; the glucose dates of twelve
subjects were repaired; SPW-Rs are the stored events with a peak of at least 5 s.d.; the correlation
type, the delta current filter, the unit of analysis and the lag range are open in the paper and
were run under every reading the claims file lists; the comodulogram uses 20 shuffles per rat.

What was found about the dataset: the glucose timestamps of twelve subjects carry dates with day
and month swapped; 27 recordings are deposited twice; the SPW-R `peak` column is relative to the
file while `start_time` is on the session clock; several SPW-R tables were made with a 4 s.d.
threshold where the Methods give 5; the session reference times are local wall-clock times labelled
UTC; there is no mapping from subjects to the paper's rats or cohorts, and no stimulation, injection,
meal or unit data; in four subjects the glucose and the electrophysiology cover different days.